# DenseNet121_HAAM_truoc_transition

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Channel Attention Module
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.fc1 = nn.Conv2d(channels, channels // reduction, kernel_size=1, bias=False)
        self.fc2 = nn.Conv2d(channels // reduction, channels, kernel_size=1, bias=False)
    
    def forward(self, x):
        avg_pool = F.adaptive_avg_pool2d(x, 1)
        max_pool = F.adaptive_max_pool2d(x, 1)
        attn = self.fc1(avg_pool + max_pool)
        attn = F.silu(attn)
        attn = self.fc2(attn)
        return torch.sigmoid(attn) * x  

# Spatial Attention Module
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=3):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, groups=1)

    def forward(self, x):
        avg_pool = torch.mean(x, dim=1, keepdim=True)
        max_pool, _ = torch.max(x, dim=1, keepdim=True)
        attn = torch.cat([avg_pool, max_pool], dim=1)
        attn = self.conv(attn)
        return torch.sigmoid(attn) * x

# Hybrid Attention Aggregation Module (HAAM)
class HAAM(nn.Module):
    def __init__(self, channels):
        super(HAAM, self).__init__()
        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()
        self.fusion_mlp = nn.Sequential(
            nn.Conv2d(2 * channels, 1, kernel_size=1),
            nn.SiLU(),
            nn.Conv2d(1, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        ca = self.ca(x)
        sa = self.sa(x)
        alpha = self.fusion_mlp(torch.cat([ca, sa], dim=1))
        M = alpha * ca + (1 - alpha) * sa
        M = M / M.max()
        return M + x

# Hàm tạo một layer Conv + BN + ReLU
def conv_layer(in_channels, out_channels, kernel_size=3, stride=1, padding=1):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

# Hàm tạo một Bottleneck Layer (BN → ReLU → Conv 1x1 → BN → ReLU → Conv 3x3 → Dropout)
def bottleneck_layer(in_channels, out_channels, dropout_rate=0.2):
    inter_channels = 4 * out_channels
    return nn.Sequential(
        nn.BatchNorm2d(in_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(in_channels, inter_channels, kernel_size=1, stride=1, padding=0, bias=False),
        nn.BatchNorm2d(inter_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(inter_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
        nn.Dropout(dropout_rate)
    )

# Hàm tạo Transition Layer (BN → ReLU → Conv 1x1 → AvgPool)
def transition_layer(in_channels, out_channels):
    return nn.Sequential(
        nn.BatchNorm2d(in_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False),
        nn.AvgPool2d(kernel_size=2, stride=2)
    )

# Hàm tạo một Dense Block với vòng lặp
def make_dense_block(in_channels, growth_rate, num_layers, dropout_rate=0.2):
    layers = []
    current_channels = in_channels
    for _ in range(num_layers):
        layer = bottleneck_layer(current_channels, growth_rate, dropout_rate)
        layers.append(layer)
        current_channels += growth_rate
    return nn.Sequential(*layers), current_channels

# DenseNet-121 với HAAM
class DenseNet121_HAAM(nn.Module):
    def __init__(self, num_classes, growth_rate=32, reduction=0.5, dropout_rate=0.2):
        super(DenseNet121_HAAM, self).__init__()

        # Số kênh ban đầu
        num_init_features = 2 * growth_rate

        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, num_init_features, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(num_init_features),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # Dense Block 1 (6 layer)
        self.dense_block1, num_features = make_dense_block(num_init_features, growth_rate, 6, dropout_rate)
        out_features = int(num_features * reduction)
        self.transition1 = transition_layer(num_features, out_features)

        # Dense Block 2 (12 layer) + HAAM
        self.dense_block2, num_features = make_dense_block(out_features, growth_rate, 12, dropout_rate)
        self.haam2 = HAAM(num_features)
        out_features = int(num_features * reduction)
        self.transition2 = transition_layer(num_features, out_features)

        # Dense Block 3 (24 layer) + HAAM
        self.dense_block3, num_features = make_dense_block(out_features, growth_rate, 24, dropout_rate)
        self.haam3 = HAAM(num_features)
        out_features = int(num_features * reduction)
        self.transition3 = transition_layer(num_features, out_features)

        # Dense Block 4 (16 layer) + HAAM
        self.dense_block4, num_features = make_dense_block(out_features, growth_rate, 16, dropout_rate)
        self.haam4 = HAAM(num_features)

        # Final BatchNorm + ReLU
        self.final_bn = nn.Sequential(
            nn.BatchNorm2d(num_features),
            nn.ReLU(inplace=True)
        )

        # Adaptive Avg Pooling
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Fully Connected Layer
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, num_classes)
        )

        # Khởi tạo trọng số
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)

        # Dense Block 1
        for layer in self.dense_block1:
            out = layer(x)
            x = torch.cat([x, out], dim=1)
        x = self.transition1(x)

        # Dense Block 2 + HAAM
        for layer in self.dense_block2:
            out = layer(x)
            x = torch.cat([x, out], dim=1)
        x = self.haam2(x)
        x = self.transition2(x)

        # Dense Block 3 + HAAM
        for layer in self.dense_block3:
            out = layer(x)
            x = torch.cat([x, out], dim=1)
        x = self.haam3(x)
        x = self.transition3(x)

        # Dense Block 4 + HAAM
        for layer in self.dense_block4:
            out = layer(x)
            x = torch.cat([x, out], dim=1)
        x = self.haam4(x)
        x = self.final_bn(x)

        x = self.avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)

        return x

# Trainning

In [ ]:
import torch
from torch import nn, save, load
from tqdm import tqdm
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torchmetrics.functional import accuracy
from torchvision.transforms import ToTensor, Resize
import numpy as np
import os
import matplotlib.pyplot as plt

# Setup CUDA
def setup_cuda():
    # Setting seeds for reproducibility
    seed = 50
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


def train_model():
    """
    Train the model over a single epoch
    :return: training loss and training accuracy
    """
    train_loss = 0.0
    train_acc = 0.0
    model.train()

    for (img, label) in tqdm(train_loader, ncols=80, desc='Training'):
        # Get a batch
        img, label = img.to(device, dtype=torch.float), label.to(device, dtype=torch.long)

        # Set the gradients to zero before starting backpropagation
        optimizer.zero_grad()

        # Perform a feed-forward pass
        logits = model(img)

        # Compute the batch loss
        loss = loss_fn(logits, label)

        # Compute gradient of the loss fn w.r.t the trainable weights
        loss.backward()

        # Update the trainable weights
        optimizer.step()

        # Accumulate the batch loss
        train_loss += loss.item()

        # Get the predictions to calculate the accuracy for every iteration. Remember to accumulate the accuracy
        prediction = logits.argmax(axis=1)
        train_acc += accuracy(prediction, label, task='multiclass', average='macro', num_classes=len(class_names)).item()

    return train_loss / len(train_loader), train_acc / len(train_loader)


def validate_model():
    """
    Validate the model over a single epoch
    :return: validation loss and validation accuracy
    """
    model.eval()
    valid_loss = 0.0
    val_acc = 0.0

    with torch.no_grad():
        for (img, label) in tqdm(val_loader, ncols=80, desc='Valid'):
            # Get a batch
            img, label = img.to(device, dtype=torch.float), label.to(device, dtype=torch.long)

            # Perform a feed-forward pass
            logits = model(img)

            # Compute the batch loss
            loss = loss_fn(logits, label)

            # Accumulate the batch loss
            valid_loss += loss.item()

            # Get the predictions to calculate the accuracy for every iteration. Remember to accumulate the accuracy
            prediction = logits.argmax(axis=1)
            val_acc += accuracy(prediction, label, task='multiclass', average='macro', num_classes=len(class_names)).item()

    return valid_loss / len(val_loader), val_acc / len(val_loader)

# Example plotting function

def plot_metrics(train_losses, val_losses, train_accuracies, val_accuracies):
    epochs = range(1, len(train_losses) + 1)
    # Losses
    plt.figure(figsize=(15, 7))
    plt.subplot(2, 1, 1)
    plt.plot(epochs, train_losses, label='Training Loss', color='blue')
    plt.plot(epochs, val_losses, label='Validation Loss', color='red')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.yscale('log')  # Log scale can help for loss curves with large values

    # Accuracies
    plt.subplot(2, 1, 2)
    plt.plot(epochs, train_accuracies, label='Training Accuracy', color='green')
    plt.plot(epochs, val_accuracies, label='Validation Accuracy', color='orange')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()

    plt.tight_layout()
    # Save the figure to a file
    plt.savefig("trainplot.png")  # You can change the file name and format (e.g., .png, .jpg, .pdf)

    plt.show()

if __name__ == "__main__":
    device = setup_cuda()

    # 1. Load the dataset
    transform = transforms.Compose([Resize((224, 224)), ToTensor()])
    train_dataset = ImageFolder(root='/kaggle/input/dataset-split/dataset_split/train', transform=transform)
    val_dataset = ImageFolder(root='/kaggle/input/dataset-split/dataset_split/val', transform=transform)
    # Get class names
    class_names = train_dataset.classes

    # 2. Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

    # 3. Create a new deep model without pre-trained weights
    model = DenseNet121_HAAM(
        num_classes=len(class_names),
    ).to(device)

    # 4. Specify loss function and optimizer
    optimizer = Adam(model.parameters(), lr=1e-4)
    loss_fn = torch.nn.CrossEntropyLoss()

    # 5. Train the model with 100 epochs
    # store the metrics for plotting
    train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []

    max_acc = 0
    for epoch in range(80):

        # 5.1. Train the model over a single epoch
        train_loss, train_acc = train_model()
        train_losses.append(train_loss)  # save train loss values
        train_accuracies.append(train_acc)  # save train acc values

        # 5.2. Validate the model after training
        val_loss, val_acc = validate_model()
        val_losses.append(val_loss)  # save val loss values
        val_accuracies.append(val_acc)  # save val acc values

        print(f'Epoch {epoch}: Train loss = {train_loss}, Train accuracy: {train_acc}')
        print(f'Epoch {epoch}: Validation loss = {val_loss}, Validation accuracy: {val_acc}')

        # 4.3. Save the model if the validation accuracy is increasing
        if val_acc > max_acc:
            print(f'Validation accuracy increased ({max_acc} --> {val_acc}). Model saved')
            folder_path = 'checkpoints_DenseNet121_HAAM'  # Define the folder name
            if not os.path.exists(folder_path):
                os.makedirs(folder_path)  # Create the folder if it does not exist
            file_path = os.path.join(folder_path,
                                     'DenseNet121_HAAM_epoch_' + str(epoch) + '_acc_{0:.4f}'.format(val_acc) + '.pt')
            with open(file_path, 'wb') as f:
                save(model.state_dict(), f)
            max_acc = val_acc
# After training is complete, plot the metrics
plot_metrics(train_losses, val_losses, train_accuracies, val_accuracies)